# Track 03 FINAL MODULE EXAM — Credit Risk Classification Pipeline
Goal: Achieve validation F1-Score >= 0.82 on imbalanced credit default predictions and submit `predictions.csv`.


In [ ]:
import io
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

TRAIN_CSV = """id,debt_ratio,credit_util,late_payments,income_k,savings_ratio,default_risk
0,0.1089,0.5391,5,97.41,0.1455,1
1,0.0883,0.5606,2,127.15,0.3185,0
2,0.8039,0.3402,2,115.45,0.0984,0
3,0.3921,0.8682,2,78.12,0.2989,0
4,0.5534,0.6183,3,106.8,0.1694,0
5,0.1381,0.1162,3,144.25,0.0785,0
6,0.6855,0.9234,3,90.16,0.3917,1
7,0.0991,0.5179,5,74.24,0.4247,0
8,0.2914,0.8878,0,39.68,0.4278,0
9,0.2738,0.2933,4,92.93,0.3835,0
10,0.3504,0.4637,5,103.84,0.4469,0
11,0.3749,0.4217,0,66.41,0.3545,0
12,0.4675,0.6082,2,121.41,0.113,0
13,0.8915,0.6534,5,109.25,0.0582,1
14,0.3839,0.6489,4,137.74,0.4393,0
15,0.3437,0.7034,0,148.06,0.046,0
16,0.6652,0.4052,4,88.68,0.0978,1
17,0.675,0.4934,2,57.47,0.3921,0
18,0.2593,0.4922,5,107.63,0.0872,1
19,0.5699,0.5748,0,39.8,0.1813,0
20,0.7544,0.8184,2,148.71,0.0637,0
21,0.1254,0.2925,4,99.69,0.095,0
22,0.8085,0.8641,3,68.2,0.0118,1
23,0.1887,0.5953,0,158.26,0.2266,0
24,0.3035,0.1911,5,101.63,0.4023,0
25,0.398,0.2058,4,91.21,0.3527,0
26,0.7352,0.5092,3,52.08,0.0333,1
27,0.1387,0.3413,5,140.72,0.0515,0
28,0.5686,0.4274,1,149.77,0.4302,0
29,0.4098,0.5061,0,87.08,0.2257,0
30,0.6597,0.3327,2,52.72,0.2916,0
31,0.223,0.9034,4,144.33,0.3331,0
32,0.5975,0.1386,5,154.38,0.1956,0
33,0.5148,0.0829,1,101.32,0.385,0
34,0.3352,0.6517,1,152.41,0.2258,0
35,0.9278,0.472,2,115.49,0.0299,0
36,0.3588,0.699,4,143.86,0.0841,0
37,0.3153,0.642,3,158.08,0.2303,0
38,0.651,0.6666,2,71.65,0.371,0
39,0.6771,0.5323,1,82.38,0.4202,0
40,0.5612,0.6883,1,93.2,0.0664,0
41,0.0819,0.3031,4,101.72,0.2943,0
42,0.1307,0.1595,4,83.02,0.4473,0
43,0.8753,0.76,4,94.25,0.3938,1
44,0.6307,0.6909,2,96.38,0.4402,0
45,0.243,0.5611,4,110.22,0.2318,0
46,0.6789,0.8955,1,41.74,0.1138,1
47,0.2609,0.9441,0,109.95,0.4099,0
48,0.7193,0.767,1,121.57,0.4249,0
49,0.5794,0.5408,4,127.46,0.4509,0
50,0.4255,0.5691,4,114.06,0.4246,0
51,0.1057,0.7393,1,95.99,0.2555,0
52,0.1705,0.8117,2,61.57,0.3866,0
53,0.6214,0.7218,3,166.24,0.1087,0
54,0.3412,0.08,4,128.58,0.1393,0
55,0.5505,0.4079,4,33.37,0.1724,1
56,0.6257,0.3234,4,92.24,0.4588,0
57,0.7441,0.488,3,56.26,0.1223,1
58,0.204,0.6636,5,90.04,0.1623,1
59,0.8621,0.6528,3,123.98,0.1207,1
60,0.4493,0.3746,5,104.36,0.4515,0
61,0.6127,0.2027,0,122.79,0.1849,0
62,0.2912,0.6866,1,143.29,0.0723,0
63,0.927,0.5311,5,95.46,0.0795,1
64,0.1609,0.1131,0,88.46,0.131,0
65,0.4775,0.3033,1,81.58,0.1289,0
66,0.2804,0.3164,5,36.13,0.2594,1
67,0.3965,0.5,4,121.72,0.0678,0
68,0.1909,0.6724,2,114.98,0.2215,0
69,0.7575,0.6454,3,97.02,0.1315,1
70,0.403,0.569,0,68.29,0.0113,0
71,0.8462,0.425,2,48.53,0.3493,0
72,0.7834,0.7083,0,135.77,0.425,0
73,0.3331,0.3474,0,67.51,0.0783,0
74,0.8462,0.4046,4,130.83,0.3417,0
75,0.4625,0.1171,4,144.27,0.2303,0
76,0.2339,0.0688,2,121.18,0.0316,0
77,0.8351,0.0597,0,85.36,0.2353,0
78,0.4815,0.4152,5,69.26,0.1477,1
79,0.0818,0.7066,4,46.74,0.1187,1
80,0.3514,0.9168,5,159.2,0.3432,0
81,0.9079,0.3049,2,105.25,0.2698,0
82,0.4747,0.9483,5,31.77,0.151,1
83,0.388,0.2351,1,106.78,0.0904,0
84,0.1036,0.166,3,30.47,0.1138,0
85,0.1754,0.1006,4,32.37,0.1385,1
86,0.7308,0.663,2,119,0.1378,0
87,0.1895,0.7205,4,70.68,0.3739,0
88,0.866,0.3716,5,83.55,0.102,1
89,0.205,0.6005,2,65.85,0.4393,0
90,0.2125,0.455,1,127.31,0.2202,0
91,0.4219,0.316,5,144.18,0.2042,0
92,0.5292,0.5221,1,132.36,0.1758,0
93,0.6234,0.513,0,159.42,0.1486,0
94,0.5712,0.4096,2,67.37,0.0452,0
95,0.4125,0.2951,5,90.69,0.2953,0
96,0.3586,0.5609,5,143.77,0.2851,0
97,0.5498,0.669,5,119.16,0.2593,1
98,0.1708,0.3457,5,65.29,0.3615,0
99,0.0924,0.9272,3,84.87,0.0827,0
100,0.8941,0.6589,0,85.45,0.0837,0
101,0.2209,0.5937,5,112.63,0.2129,0
102,0.5083,0.1819,2,133.67,0.0394,0
103,0.0824,0.3795,4,28.81,0.4248,0
104,0.1228,0.1807,1,141.88,0.0972,0
105,0.508,0.3991,0,58.19,0.379,0
106,0.3674,0.2339,3,30.29,0.2928,0
107,0.2776,0.0657,0,150,0.3302,0
108,0.1038,0.9193,0,131.84,0.3794,0
109,0.6131,0.1185,0,155.93,0.0579,0
110,0.3807,0.7867,2,125.87,0.2483,0
111,0.4603,0.1673,4,95.67,0.2629,0
112,0.5783,0.8942,0,157.95,0.2836,0
113,0.3633,0.6084,4,71.96,0.1265,1
114,0.4922,0.4938,4,112.15,0.158,0
115,0.12,0.3599,3,157.28,0.2202,0
116,0.8185,0.2161,2,133.38,0.4328,0
117,0.3679,0.7114,2,86.88,0.1112,0
118,0.6723,0.7687,4,95.01,0.3472,1
119,0.2534,0.0548,1,151.18,0.4109,0
120,0.2639,0.913,0,166.66,0.0558,0
121,0.0992,0.0984,3,92.69,0.0501,0
122,0.6103,0.1103,0,50.74,0.0947,0
123,0.7161,0.0865,0,33.55,0.2286,0
124,0.1463,0.5904,0,111.28,0.1065,0
125,0.6406,0.9227,0,38.82,0.1101,0
126,0.3534,0.5314,3,48.43,0.0299,1
127,0.9041,0.7622,3,118.18,0.2959,0
128,0.8042,0.6222,1,82.23,0.4173,0
129,0.4016,0.3642,4,77.43,0.4152,0
130,0.9196,0.7749,0,117.2,0.4019,0
131,0.5842,0.2392,4,58.83,0.0668,1
132,0.4033,0.7167,5,82.59,0.3864,1
133,0.3173,0.7999,0,133.29,0.2579,0
134,0.7829,0.8022,0,29.78,0.2652,0
135,0.317,0.8776,3,36.91,0.0126,1
136,0.8296,0.8987,3,104.92,0.0755,1
137,0.4769,0.2458,2,160.43,0.12,0
138,0.338,0.9214,2,77.14,0.35,0
139,0.804,0.3957,5,152.72,0.4086,0
140,0.7918,0.8363,0,86.01,0.0305,0
141,0.1221,0.6357,1,154.4,0.2722,0
142,0.3624,0.3522,4,86.95,0.172,0
143,0.6979,0.7011,5,119.25,0.1205,1
144,0.2709,0.944,1,64.67,0.37,0
145,0.7133,0.5972,0,102.49,0.0223,0
146,0.225,0.5216,0,47.14,0.3706,0
147,0.1559,0.0862,1,59.9,0.0606,0
148,0.4628,0.3655,3,52.84,0.1162,0
149,0.8393,0.9526,4,96.56,0.2781,1
150,0.1716,0.8702,2,50.46,0.367,0
151,0.8294,0.4084,1,112.01,0.4094,0
152,0.6418,0.5313,2,116.03,0.1781,0
153,0.3462,0.2999,2,74.51,0.3476,0
154,0.9244,0.5284,1,39.76,0.2991,0
155,0.7044,0.1228,2,52.99,0.4418,0
156,0.1754,0.0931,3,108.18,0.1547,0
157,0.4462,0.2375,0,165.72,0.4153,0
158,0.8844,0.0793,2,93.9,0.4599,0
159,0.8804,0.4671,0,156.56,0.2129,0
160,0.1175,0.2821,5,138.59,0.0355,0
161,0.5051,0.8908,4,62.97,0.0159,1
162,0.2502,0.8055,0,109.77,0.318,0
163,0.6952,0.7142,1,115.07,0.4145,0
164,0.6021,0.7715,3,82.39,0.0479,1
165,0.1815,0.6615,1,147.02,0.024,0
166,0.1074,0.3213,2,60.32,0.3801,0
167,0.3799,0.8405,1,111.82,0.3828,0
168,0.8779,0.7097,2,132.73,0.0186,0
169,0.1914,0.1671,4,128.79,0.165,0
170,0.4259,0.1146,5,40.21,0.2899,1
171,0.6381,0.7171,3,44.13,0.2322,1
172,0.6522,0.3828,4,134.09,0.1664,0
173,0.2285,0.8319,5,84.5,0.1746,1
174,0.2638,0.5679,1,161.57,0.14,0
175,0.6139,0.2221,3,108.18,0.0363,0
176,0.4236,0.2267,4,115.92,0.1153,0
177,0.8699,0.5246,2,30.96,0.142,1
178,0.1051,0.4172,3,77.02,0.1204,0
179,0.4955,0.3306,1,104.14,0.2535,0
180,0.6727,0.6688,0,90.86,0.0974,0
181,0.916,0.7444,5,126.94,0.1866,1
182,0.1292,0.7939,3,43.28,0.2366,0
183,0.7397,0.8043,0,47.4,0.0511,1
184,0.7085,0.6721,3,83.24,0.4107,0
185,0.2902,0.9301,5,77.33,0.1746,1
186,0.5709,0.08,5,39.25,0.0714,1
187,0.4552,0.4327,4,84.76,0.1114,0
188,0.4091,0.9197,4,143.52,0.2888,0
189,0.4214,0.9355,1,92.49,0.2125,0
190,0.6674,0.9262,3,36.07,0.3651,1
191,0.799,0.3438,1,33.44,0.0604,0
192,0.7239,0.798,5,116.64,0.1136,1
193,0.1696,0.8203,0,55.78,0.0311,0
194,0.2457,0.5486,5,130.99,0.2564,0
195,0.676,0.4227,0,111.73,0.2064,0
196,0.6584,0.8749,0,73.27,0.3796,0
197,0.644,0.378,2,64.93,0.2068,0
198,0.5477,0.9584,4,118.03,0.2358,1
199,0.737,0.7672,2,49.77,0.4035,1
200,0.7378,0.3046,0,35.7,0.0358,0
201,0.8574,0.3111,2,42.17,0.4185,0
202,0.0943,0.5216,2,140.42,0.4242,0
203,0.6133,0.8876,3,106.32,0.3556,0
204,0.8401,0.108,1,122.33,0.0187,0
205,0.8703,0.1706,5,111.42,0.1076,1
206,0.1217,0.2568,5,134.39,0.3084,0
207,0.5206,0.2679,4,57.85,0.0515,1
208,0.4524,0.8805,4,90.12,0.0307,1
209,0.4295,0.1657,4,162.93,0.2105,0
210,0.9047,0.2068,0,58.99,0.4055,0
211,0.6627,0.689,0,52.9,0.3146,0
212,0.3658,0.3205,0,147.05,0.4259,0
213,0.1752,0.3185,1,160.68,0.1341,0
214,0.7482,0.3537,1,52.53,0.3703,0
215,0.4156,0.639,4,75.42,0.4005,1
216,0.8523,0.4858,0,75.13,0.3756,0
217,0.6227,0.6241,2,91.28,0.216,0
218,0.5004,0.0602,2,157.66,0.2492,0
219,0.2119,0.939,5,51.08,0.0755,1
220,0.8717,0.8613,1,135.05,0.4238,0
221,0.6547,0.7146,5,49.29,0.0169,1
222,0.6121,0.7121,4,103.39,0.4556,0
223,0.9204,0.1472,0,117.32,0.0971,0
224,0.2924,0.4271,3,45.01,0.0547,0
225,0.1462,0.6908,1,51.25,0.3875,0
226,0.4638,0.0868,2,122.18,0.1369,0
227,0.3981,0.913,2,117.91,0.3819,0
228,0.8686,0.1839,0,131.92,0.1339,0
229,0.8463,0.52,0,123.07,0.0143,0
230,0.6013,0.6981,4,137.46,0.32,0
231,0.8911,0.6878,2,122.88,0.3498,0
232,0.1975,0.7263,3,28.04,0.3803,0
233,0.272,0.4554,2,46.33,0.3082,0
234,0.9223,0.4192,5,75.97,0.1321,1
235,0.8354,0.9698,4,90.92,0.3907,1
236,0.8961,0.2208,1,164.51,0.4077,0
237,0.2723,0.0966,5,46.79,0.2162,1
238,0.4137,0.3234,2,142.81,0.4288,0
239,0.3089,0.6173,3,139.81,0.4377,0
240,0.495,0.4053,3,60.8,0.1447,0
241,0.2591,0.3147,1,111.47,0.3934,0
242,0.9211,0.1746,3,134.55,0.1212,0
243,0.3141,0.2033,0,102.62,0.1124,0
244,0.2996,0.786,5,135.23,0.3483,0
245,0.1899,0.0614,3,95.78,0.2844,0
246,0.5857,0.3666,5,149.34,0.1803,0
247,0.6647,0.8255,4,60.45,0.4063,1
248,0.153,0.3297,0,162.7,0.1955,0
249,0.7363,0.1159,3,81.49,0.2257,0
250,0.7033,0.908,2,138.83,0.1075,0
251,0.18,0.364,3,64.93,0.1374,0
252,0.792,0.4248,2,121.1,0.274,0
253,0.1946,0.857,2,122.27,0.2527,0
254,0.5687,0.1857,2,131.47,0.2615,0
255,0.6161,0.5799,3,81.93,0.0229,1
256,0.7303,0.3194,4,127.5,0.1475,1
257,0.6204,0.4289,1,51.4,0.2284,0
258,0.7978,0.4202,3,57.18,0.3865,1
259,0.6047,0.8362,3,108.06,0.4021,0
260,0.1926,0.1209,4,51.94,0.1073,0
261,0.5875,0.7233,2,90.16,0.1886,0
262,0.1978,0.1538,2,48.55,0.1617,0
263,0.4815,0.898,1,39.74,0.4221,0
264,0.5808,0.692,4,84,0.1301,1
265,0.3665,0.399,2,109.1,0.0363,0
266,0.7728,0.759,1,29.12,0.272,0
267,0.1421,0.2758,3,41.87,0.2149,0
268,0.2571,0.1207,3,122.13,0.3727,0
269,0.5412,0.2935,0,63.84,0.1707,0
270,0.4788,0.7074,3,129.6,0.1539,0
271,0.8712,0.2462,2,37.35,0.0753,1
272,0.2028,0.7731,5,155.3,0.0529,1
273,0.8921,0.6612,4,69.18,0.306,1
274,0.1618,0.6103,0,66.47,0.2222,0
275,0.3611,0.8505,5,40.04,0.4384,1
276,0.4924,0.9112,4,73.57,0.2096,1
277,0.1457,0.5605,3,105.88,0.3706,0
278,0.4181,0.7927,2,111.59,0.1534,0
279,0.7191,0.6764,5,76.74,0.4192,1
280,0.9237,0.734,1,72.21,0.1106,1
281,0.1915,0.129,5,89.15,0.1655,0
282,0.6351,0.4121,4,38.24,0.019,1
283,0.2563,0.6377,0,141.76,0.1756,0
284,0.1141,0.7525,2,114.53,0.1238,0
285,0.691,0.1576,4,58.78,0.3064,1
286,0.8247,0.9659,2,59.91,0.041,1
287,0.1271,0.7321,1,125.62,0.124,0
288,0.3929,0.0694,0,115.41,0.3315,0
289,0.4902,0.1339,3,68.94,0.222,0
290,0.69,0.4106,2,72.65,0.3808,0
291,0.5741,0.527,3,51.5,0.2624,1
292,0.7447,0.3675,2,116.93,0.3612,0
293,0.8279,0.9566,5,31.95,0.4533,1
294,0.5599,0.626,4,134.98,0.3344,0
295,0.1437,0.8971,0,161.3,0.0318,0
296,0.5932,0.8023,1,72.66,0.0684,0
297,0.1259,0.625,5,106.62,0.2386,1
298,0.7488,0.4507,3,101.38,0.108,1
299,0.8345,0.5898,1,67.01,0.3918,0
300,0.1774,0.7388,0,133.09,0.4276,0
301,0.4581,0.9216,2,106.56,0.0868,0
302,0.5967,0.8692,5,122.95,0.1947,1
303,0.6815,0.574,5,64.18,0.0742,1
304,0.4513,0.2536,0,37.4,0.254,0
305,0.1968,0.7592,3,119.18,0.201,0
306,0.1495,0.522,5,55.27,0.2339,1
307,0.6221,0.3103,1,148.81,0.0919,0
308,0.5216,0.6914,5,61.67,0.2169,1
309,0.2291,0.8456,2,79.27,0.4282,0
310,0.4873,0.658,3,120.52,0.0134,0
311,0.4566,0.8133,5,129.26,0.2054,1
312,0.3533,0.2078,1,76.36,0.3746,0
313,0.5153,0.8252,5,144.56,0.4412,0
314,0.8178,0.4966,1,75.3,0.098,0
315,0.0989,0.3271,2,71.88,0.4335,0
316,0.5002,0.6354,5,130.06,0.4376,1
317,0.4925,0.8255,5,89.69,0.1085,1
318,0.5761,0.3299,2,123.49,0.1173,0
319,0.7066,0.589,1,83.05,0.2421,0
320,0.2963,0.6698,4,55.43,0.0701,1
321,0.8631,0.3948,3,78.11,0.0763,1
322,0.8529,0.7903,3,29.29,0.0367,1
323,0.1477,0.9067,2,115.74,0.3058,0
324,0.51,0.2105,1,56.26,0.166,0
325,0.6284,0.2851,3,146.96,0.2724,0
326,0.2575,0.7753,5,133.61,0.0318,1
327,0.2184,0.2523,2,105.37,0.4534,0
328,0.4878,0.2525,2,120.21,0.0675,0
329,0.3365,0.6554,5,56.83,0.2665,1
330,0.8365,0.1227,1,107.69,0.3058,0
331,0.5499,0.1348,2,111.9,0.0847,0
332,0.5389,0.3115,5,57.7,0.0789,1
333,0.0837,0.4229,4,29.97,0.176,1
334,0.5832,0.1388,3,101.91,0.2572,0
335,0.5457,0.6409,5,81.83,0.2747,1
336,0.5381,0.6959,2,140.88,0.3732,0
337,0.385,0.6467,5,141.48,0.2125,0
338,0.3242,0.4125,1,119.71,0.1599,0
339,0.269,0.2101,2,33.27,0.2612,0
340,0.3232,0.7322,4,58.39,0.3692,1
341,0.6957,0.4746,0,51.15,0.3731,0
342,0.9184,0.194,5,28.58,0.248,1
343,0.4017,0.1237,2,38.61,0.0788,0
344,0.3913,0.4937,0,37.71,0.2816,0
345,0.183,0.4993,2,144.22,0.1539,0
346,0.4626,0.3598,3,73.89,0.0349,1
347,0.6987,0.5214,3,46.92,0.1151,1
348,0.5621,0.8872,5,125.27,0.0894,1
349,0.6979,0.703,2,107.82,0.1873,0
350,0.7108,0.6658,4,154.16,0.3627,0
351,0.2826,0.2725,5,83.69,0.1793,0
352,0.4231,0.4464,3,122.77,0.3996,0
353,0.3977,0.5782,2,56.88,0.394,0
354,0.5862,0.8036,0,95.18,0.2856,0
355,0.4687,0.6904,4,128.19,0.0907,0
356,0.7622,0.1383,2,101.88,0.0309,0
357,0.2547,0.9464,0,128.83,0.0248,0
358,0.1208,0.0665,2,151.74,0.2485,0
359,0.3082,0.8114,3,165.83,0.4376,0"""
TEST_CSV = """id,debt_ratio,credit_util,late_payments,income_k,savings_ratio
0,0.1899,0.9241,1,67.91,0.284
1,0.8717,0.5067,3,129.87,0.4312
2,0.7976,0.2324,2,108.12,0.3684
3,0.638,0.2324,3,75.01,0.0622
4,0.1595,0.8577,5,139.34,0.0417
5,0.2402,0.2214,3,31.11,0.0876
6,0.8614,0.7219,5,140.84,0.1404
7,0.2268,0.6232,5,145.54,0.2389
8,0.5593,0.9137,4,75.91,0.0575
9,0.1232,0.8148,1,118.17,0.1274
10,0.4805,0.0661,3,107.9,0.0752
11,0.3902,0.7006,4,124.65,0.0113
12,0.6676,0.4734,1,68.22,0.3956
13,0.1163,0.2049,5,78.63,0.2255
14,0.9007,0.7968,2,42.38,0.0891
15,0.3759,0.9155,2,104.52,0.3474
16,0.3914,0.5593,3,29.26,0.2427
17,0.9076,0.0895,5,155.33,0.32
18,0.5353,0.4216,4,165.39,0.1299
19,0.1028,0.6303,3,161.02,0.1669
20,0.47,0.7934,3,165.86,0.2433
21,0.5128,0.4578,3,127.71,0.0468
22,0.8335,0.2952,5,74.09,0.2616
23,0.6394,0.8891,2,57.5,0.1166
24,0.6118,0.7921,4,32.45,0.106
25,0.8187,0.5998,0,55.28,0.0606
26,0.3515,0.4998,0,139.69,0.1813
27,0.1956,0.5989,3,76.15,0.4246
28,0.4561,0.3907,4,140.82,0.4379
29,0.3827,0.7784,5,97.88,0.0564
30,0.4161,0.4046,3,99.53,0.1209
31,0.1726,0.9405,4,117.68,0.2541
32,0.2951,0.9135,1,161.79,0.0565
33,0.5134,0.7117,3,167.78,0.1225
34,0.4084,0.0755,3,161.11,0.2533
35,0.7916,0.7571,5,152.64,0.1444
36,0.1474,0.1087,0,42.14,0.4458
37,0.3969,0.7201,0,60.72,0.0381
38,0.673,0.0849,5,119.76,0.326
39,0.3155,0.4374,2,157.65,0.3978
40,0.1705,0.1988,3,97.87,0.145
41,0.1892,0.9477,5,147.59,0.4342
42,0.4963,0.529,0,146.21,0.1691
43,0.6832,0.2955,4,59.1,0.0922
44,0.3452,0.5634,5,98.89,0.3793
45,0.5606,0.4773,2,46.17,0.2856
46,0.2211,0.1954,2,145.66,0.2023
47,0.3722,0.5642,2,44.56,0.0566
48,0.4226,0.9278,4,145.76,0.2592
49,0.8453,0.4089,2,46.83,0.0579
50,0.6156,0.8671,5,103.04,0.4404
51,0.2928,0.8892,2,78.78,0.2531
52,0.8099,0.1338,2,143.96,0.3431
53,0.3356,0.1775,3,75.44,0.2721
54,0.5271,0.4985,1,53.96,0.2915
55,0.8843,0.5485,5,56.23,0.3702
56,0.8917,0.7848,2,106.72,0.2154
57,0.3585,0.6541,4,32.55,0.0678
58,0.3483,0.1559,4,80.57,0.0505
59,0.7853,0.9295,2,126.8,0.4323
60,0.7772,0.8339,0,166.66,0.0316
61,0.1092,0.3685,0,114.23,0.4287
62,0.7524,0.7948,2,52.51,0.2768
63,0.756,0.2992,5,112.38,0.3287
64,0.4696,0.3112,5,81.49,0.3933
65,0.7464,0.8051,4,29.82,0.4023
66,0.8875,0.5257,4,78.22,0.1543
67,0.6929,0.3216,5,71.89,0.4556
68,0.6501,0.2091,3,134.28,0.2722
69,0.5022,0.6827,1,112.65,0.2246
70,0.5094,0.5046,2,38.93,0.2298
71,0.1187,0.7929,4,50.87,0.2002
72,0.4136,0.8628,4,99.76,0.3382
73,0.4347,0.7456,0,63.6,0.388
74,0.7071,0.712,4,38.78,0.2487
75,0.4311,0.9093,0,92.81,0.3391
76,0.383,0.6457,4,85.61,0.1876
77,0.1235,0.4714,3,158.38,0.1066
78,0.6124,0.6833,0,163.82,0.3615
79,0.5041,0.8114,3,52.98,0.4514
80,0.7094,0.7977,4,106.03,0.3337
81,0.4114,0.133,0,148.67,0.1124
82,0.6075,0.6725,0,31.85,0.3881
83,0.6592,0.0631,3,77.74,0.4351
84,0.5226,0.5639,5,126.65,0.1957
85,0.4024,0.8841,3,162.94,0.4346
86,0.2311,0.3955,5,63.47,0.162
87,0.4581,0.2325,1,85.37,0.0455
88,0.2014,0.4241,2,141.59,0.0929
89,0.2185,0.8588,1,89.05,0.3637
90,0.2432,0.835,4,56.84,0.1407
91,0.2401,0.0598,5,98.03,0.4416
92,0.909,0.3013,2,146.51,0.2181
93,0.1857,0.2533,4,74.33,0.2253
94,0.4142,0.4487,1,122.74,0.1317
95,0.1366,0.3157,0,122.93,0.4487
96,0.6837,0.7543,2,75.45,0.1725
97,0.6784,0.1932,3,52.4,0.2051
98,0.8785,0.1545,1,94.04,0.228
99,0.2972,0.3313,2,120.72,0.062
100,0.5295,0.1212,5,115.92,0.3454
101,0.7671,0.0992,1,80.93,0.1888
102,0.2062,0.6538,0,90.9,0.3625
103,0.3028,0.1529,5,71.04,0.1741
104,0.4013,0.6127,1,147.38,0.4486
105,0.4585,0.57,5,121.88,0.1871
106,0.8653,0.9258,4,62.86,0.3273
107,0.1727,0.6257,4,101.78,0.2731
108,0.1386,0.7584,0,81,0.1913
109,0.7101,0.5593,0,107.14,0.2682
110,0.2781,0.6904,5,80.59,0.3232
111,0.3451,0.1565,1,30.01,0.2239
112,0.191,0.8836,4,87.14,0.2207
113,0.1506,0.0907,1,163.28,0.0855
114,0.5253,0.6207,2,127.44,0.1385
115,0.8971,0.7295,4,73.46,0.3351
116,0.4283,0.0589,1,72.91,0.4067
117,0.5319,0.1427,5,79.9,0.4184
118,0.6953,0.8131,0,50.48,0.2337
119,0.3606,0.8986,0,95.78,0.0499"""

train = pd.read_csv(io.StringIO(TRAIN_CSV))
test_X = pd.read_csv(io.StringIO(TEST_CSV))
print("Class distribution:\n", train["default_risk"].value_counts())


In [ ]:
features = ["debt_ratio", "credit_util", "late_payments", "income_k", "savings_ratio"]
X_tr, y_tr = train[features], train["default_risk"]

# TODO: Compare LogisticRegression vs RandomForestClassifier vs SVC
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, random_state=42))
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_cv = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="f1").mean()
print(f"5-Fold CV F1-Score: {f1_cv:.4f}")

pipe.fit(X_tr, y_tr)
preds = pipe.predict(test_X[features])
pd.DataFrame({"id": test_X["id"], "prediction": preds}).to_csv("predictions.csv", index=False)
print("Saved predictions.csv:", len(preds), "rows")
